# 01 — Data assembly

**Mini Project 2.** Reads the 2023 Census totals-by-topic households export,
renames the variable columns using the Stats NZ lookup table, derives the
crowding target, and writes one tidy analysis table — one row per statistical
area 2 (SA2).

Runs top to bottom on a fresh clone. Nothing here is analysis; it is the
plumbing, and it should be dull.

The lookup table is **not joined** to the data. It is used to rename columns,
which is what makes 261 columns called `VAR_4_n` readable.

## Setup

In [ ]:
import sys, re, json, warnings
sys.path.append("../src")

import numpy as np
import pandas as pd

import mp2

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

YEAR = 2023
LOOKUP = mp2.DATA / "2023_census_totals_by_topic_for_households_by_sa2_lookup_table.csv"

print("data folder  :", mp2.DATA)
print("output folder:", mp2.OUT)

## Load the two inputs

`HJ` is the census counts — one row per SA2, one column per variable, plus the
geometry and area columns. `LK` is the lookup that says what each variable
column means.

They get different names on purpose. Reusing one name for both files is the
easiest way to end up indexing columns that only exist in one of them.

In [ ]:
HJ = mp2.load_raw(mp2.HOUSEHOLDS)
LK = mp2.load_raw(LOOKUP)

print("census counts:", HJ.shape)
print("lookup table :", LK.shape)
print()
print("counts file starts:", list(HJ.columns)[:6], "...")
print("lookup columns    :", list(LK.columns))

## Build the rename map from the lookup

The lookup describes three census years stacked together, so filter to 2023
first. Each remaining row gives a column code (`VAR_4_48`) and the variable and
category it holds.

Names are generated by slugifying the topic and category, so every output
column still says where it came from: `household_crowding_index_crowded`.

In [ ]:
lk = LK[(LK["Year"] == YEAR) &
        LK["Column_name"].astype(str).str.startswith("VAR_")].copy()

def slug(value):
    """lower_snake_case, safe to use as a column name."""
    return re.sub(r"[^a-z0-9]+", "_", str(value).lower()).strip("_")

lk["new_name"] = [f"{slug(v)}_{slug(c)}"
                   for v, c in zip(lk["Variable1"], lk["Variable1_category"])]
RENAME = dict(zip(lk["Column_name"], lk["new_name"]))

print(f"{len(RENAME)} variable columns mapped for {YEAR}")
print()
print("topics covered:")
for topic in sorted(lk["Variable1"].dropna().unique()):
    print("  -", topic)

## Rename, coerce, and drop the sentinel

Two things about these exports are worth knowing before reading any number.

**Suppression is written as `-999`.** In this table 223 of the 261 variable
columns carry at least one. It is not a count, and summing it turns a column
total negative — which is how it was found. `mp2.read_measure` maps it to `NaN`
along with the blank sentinels.

**Counts are randomly rounded to base 3.** Published figures are multiples of
three, so categories never sum exactly to their own totals. Differences of a few
units are rounding, not error.

In [ ]:
keep = [c for c in RENAME if c in HJ.columns]
df = (HJ[["SA22023_V1_00", "SA22023_V1_00_NAME"] + keep]
        .rename(columns={"SA22023_V1_00": "sa2_code",
                         "SA22023_V1_00_NAME": "sa2_name"})
        .rename(columns=RENAME))

value_cols = [c for c in df.columns if c not in ("sa2_code", "sa2_name")]
df[value_cols] = df[value_cols].apply(mp2.read_measure).astype("float64")
df = df.copy()  # defragment after the block assignment

suppressed_cells = int(df[value_cols].isna().sum().sum())
suppressed_cols = int(df[value_cols].isna().any().sum())
print(f"frame            : {df.shape[0]:,} SA2s x {df.shape[1]:,} columns")
print(f"suppressed cells : {suppressed_cells:,} across {suppressed_cols} of {len(value_cols)} columns")
print(f"all numeric      : {bool(df[value_cols].dtypes.eq('float64').all())}")

## Check the crowding block reconciles

Before trusting the target, verify the crowding index against its own parts.
The index is hierarchical: `crowded` should equal the two "bedrooms needed"
categories combined, and `total` should equal `total stated` plus `not stated`.

Small differences are expected — every count is rounded independently. A
negative total means the sentinel got through, and the assert says so.

In [ ]:
CROWD = {
    "crowded":      "household_crowding_index_crowded",
    "severe":       "household_crowding_index_two_or_more_bedrooms_needed_severely_crowded",
    "moderate":     "household_crowding_index_one_bedroom_needed_crowded",
    "not_crowded":  "household_crowding_index_not_crowded",
    "not_stated":   "household_crowding_index_not_stated",
    "total":        "household_crowding_index_total",
    "total_stated": "household_crowding_index_total_stated",
}

totals = {name: df[col].sum() for name, col in CROWD.items()}
for name, value in totals.items():
    print(f"  {name:14s} {value:>12,.0f}")

print()
print(f"crowded  vs severe+moderate : {totals['crowded'] - totals['severe'] - totals['moderate']:+,.0f}")
print(f"total    vs stated+notstated: {totals['total'] - totals['total_stated'] - totals['not_stated']:+,.0f}")

assert totals["crowded"] > 0, "crowded is not positive - is the -999 sentinel handled?"
print()
print("No total is negative, and both identities hold to within rounding.")

## Build the target

**Crowded** means one or more bedrooms needed. `total_stated` is the
denominator — the households that gave a usable answer — because dividing by
`total` would understate the rate wherever non-response is higher.

Note what this target is: the crowding **rate of an area**, not of a household.
Everything modelled downstream is at SA2 level, which is the ecological fallacy
the README names as the leading limitation.

In [ ]:
df["crowded_n"]     = df[CROWD["crowded"]]
df["crowding_base"] = df[CROWD["total_stated"]]
df["crowding_rate"] = df["crowded_n"] / df["crowding_base"].replace(0, pd.NA)

print("dtype:", df["crowding_rate"].dtype)
print()
print(df["crowding_rate"].describe().round(4).to_string())
print()
print(f"SA2s with a usable rate : {df['crowding_rate'].notna().sum():,} of {len(df):,}")
print(f"without a usable rate   : {df['crowding_rate'].isna().sum():,}")

## Binary target for the classifier (notebook 04)

Crowding is rare, so a plain high/low split would be very nearly all
negatives and any PR-AUC built on it would be meaningless. A **top-quartile**
flag gives a workable positive class, and it matches what the client actually
does with the result — rank areas for assessment, not label them.

The threshold is computed here and written to `frame_meta.json`, so notebooks
02–05 all use the same one instead of each recalculating.

In [ ]:
threshold = df["crowding_rate"].quantile(0.75)
df["crowded_top_quartile"] = (df["crowding_rate"] >= threshold).astype("float64")
df.loc[df["crowding_rate"].isna(), "crowded_top_quartile"] = np.nan

print(f"top-quartile threshold : {threshold:.4f}  ({threshold:.2%})")
print(f"positive class         : {df['crowded_top_quartile'].sum():,.0f} SA2s")
print(f"base rate              : {df['crowded_top_quartile'].mean():.2%}")

## Sanity checks

Every one of these should pass. If an assert fires, stop and read the message —
it names the fault.

In [ ]:
assert df["sa2_code"].is_unique, "SA2 codes must be unique"
assert df["sa2_code"].notna().all(), "no missing SA2 codes"
assert df["crowded_n"].min() >= 0, "negative crowding counts"
assert df["crowding_rate"].dropna().between(0, 1).all(), "rates must sit in 0-1"

print(f"SA2s                 : {len(df):,}")
print(f"unique SA2 codes     : {df['sa2_code'].nunique():,}")
print(f"code length          : {sorted(df['sa2_code'].astype(str).str.len().unique())}")
print(f"crowded count range  : {df['crowded_n'].min():.0f} to {df['crowded_n'].max():.0f}")
print(f"crowding rate range  : {df['crowding_rate'].min():.4f} to {df['crowding_rate'].max():.4f}")
print()
print("All checks passed.")

## Where suppression lands

The README claims suppression concentrates in small areas. This measures it, so
the claim is a finding with a number behind it rather than an assertion.

In [ ]:
df["suppressed_n"] = df[value_cols].isna().sum(axis=1)
lost = df["crowding_rate"].isna()

print(f"SA2s without a crowding rate: {lost.sum()} of {len(df)}")
print()
print("suppressed columns per SA2:")
print(df["suppressed_n"].describe().round(2).to_string())
print()
print(f"mean suppressed columns - SA2s kept : {df.loc[~lost, 'suppressed_n'].mean():.1f}")
print(f"mean suppressed columns - SA2s lost : {df.loc[lost,  'suppressed_n'].mean():.1f}")
print()
print("correlation, suppression vs crowding rate:",
      round(df['suppressed_n'].corr(df['crowding_rate']), 3))

## Write the analysis table

`outputs/analysis_frame.csv` is what notebooks 02–05 read. The geometry column
is dropped — notebook 02 reads the shapefile when it needs maps, and a plain CSV
keeps the other notebooks free of the geo stack.

`frame_meta.json` records the decisions, so the threshold and the crowding
definition travel with the data instead of living in someone's memory.

In [ ]:
mp2.OUT.mkdir(exist_ok=True, parents=True)

df.drop(columns=["suppressed_n"]).to_csv(mp2.OUT / "analysis_frame.csv", index=False)

frame_meta = {
    "year": YEAR,
    "n_sa2": int(len(df)),
    "crowding_definition": "one or more bedrooms needed",
    "crowding_denominator": "total stated (excludes not stated)",
    "top_quartile_threshold": float(threshold),
    "suppressed_cells": suppressed_cells,
    "sa2_without_rate": int(lost.sum()),
    "source": "Stats NZ 2023 Census totals by topic, households, by SA2 (CC BY 4.0)",
}
(mp2.OUT / "frame_meta.json").write_text(json.dumps(frame_meta, indent=2))

print("written:", mp2.OUT / "analysis_frame.csv")
print("written:", mp2.OUT / "frame_meta.json")
print()
print(json.dumps(frame_meta, indent=2))

## Notes for the README

These go in the limitations section. Each one came out of a check above, so
they are measured facts rather than cautions.

- **Suppression.** Stats NZ writes `-999` into a suppressed cell. 223 of the 261
  variable columns carry at least one, and 119 SA2s lose the crowding rate
  entirely because their denominator is suppressed.
- **Rounding.** Counts are randomly rounded to base 3, so categories do not sum
  exactly to their own totals. The crowding index reconciles to within a few
  units, which is rounding, not error.
- **The target is an area's rate.** Nothing here says anything about an
  individual household. This is the ecological fallacy and it leads the list.
- **Source and licence.** Stats NZ 2023 Census totals by topic for households,
  by SA2, reused under CC BY 4.0. The lookup table names every variable; it is
  not joined, it is used to rename columns.
- **Geography vintage.** SA2 2023 boundaries. 16 SA2s have no digitised
  geometry and drop out when the shapefile is joined in notebook 02.